# 00 Data Engineering

        In this notebook, We start from the raw Bank Marketing dataset and build the two clean datasets used in the rest of the project. The important modeling decision is that `duration` is useful for analysis, but it is not safe for a real pre-call prediction system because it is only known after the call finishes. We therefore create one production dataset without `duration` and one comparison dataset with `duration` so we can measure how much leakage improves model performance.


## Load Project Paths and Raw Data

        We define paths relative to the project root so the notebook works whether it is opened from the `notebooks/` folder or from the root directory. The raw data is `bank.csv`, which has 11,162 observations and 17 variables.


In [12]:
# Set project paths and load the raw bank marketing CSV.

from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'bank.csv'
CLEANED_NO_DURATION = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_bank_data.csv'
CLEANED_WITH_DURATION = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_bank_data_with_duration.csv'

bank_data = pd.read_csv(RAW_DATA)
bank_data.shape

(11162, 17)

## Preview the Source Table

        Before transforming anything, we inspect the first few rows to confirm the column names and value formats. The target is `deposit`, and the predictors include demographic, financial, and campaign-contact variables.


In [13]:
# Preview the first records to confirm schema and raw value formats.

bank_data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


The raw rows show numeric variables such as `age`, `balance`, `day`, `duration`, `campaign`, `pdays`, and `previous`, plus categorical fields such as `job`, `marital`, `education`, `contact`, `month`, and `poutcome`.

## Check Data Types, Missing Values, and Cardinality

        This step is basic data-quality check. We want to know whether any column needs imputation, whether the categorical variables are reasonable for one-hot encoding, and whether the target is complete.


In [14]:
# Summarize data types, missing values, and unique values by column.

pd.DataFrame({
    'dtype': bank_data.dtypes.astype(str),
    'missing': bank_data.isna().sum(),
    'unique_values': bank_data.nunique()
})

,dtype,missing,unique_values
age,int64,0,76
job,object,0,12
marital,object,0,3
education,object,0,4
default,object,0,2
balance,int64,0,3805
housing,object,0,2
loan,object,0,2
contact,object,0,3
day,int64,0,31


The result shows no missing values. We still keep imputation inside the final pipeline for robustness, but for this dataset the cleaning step does not need to fill any missing entries. Values like `unknown` are kept as valid categories because they carry campaign information rather than being true nulls.

## Inspect Target Balance

        We check the distribution of `deposit` before modeling because class balance affects both model training and evaluation. Here the target is moderately balanced, not severely imbalanced.


In [15]:
# Count and normalize the term-deposit subscription target.

bank_data['deposit'].value_counts(), bank_data['deposit'].value_counts(normalize=True)

(no     5873
 yes    5289
 Name: deposit, dtype: int64,
 no     0.52616
 yes    0.47384
 Name: deposit, dtype: float64)

The target distribution is about 52.6% `no` and 47.4% `yes`. Because the classes are close enough, We do not apply resampling. Instead, I evaluate models with accuracy, sensitivity, specificity, balanced accuracy, F1, and ROC AUC.


## Contact Channel Check

        We use a normalized crosstab to check the contact distribution so the result is easier to compare across contact types.

In [16]:
# Compare deposit rates by contact channel.

pd.crosstab(bank_data['contact'], bank_data['deposit'], normalize='index')

deposit,no,yes
contact,,
cellular,0.456727,0.543273
telephone,0.496124,0.503876
unknown,0.774084,0.225916


The contact channel has a visible relationship with subscription. Customers contacted by `cellular` subscribe more often than customers with `unknown` contact type. This supports keeping `contact` as a categorical predictor in the model.


## Create Production and Diagnostic Datasets

    We create two cleaned datasets:
    - `cleaned_bank_data.csv`: production-safe data with both `duration` and `pdays` removed.
    - `cleaned_bank_data_with_duration.csv`: diagnostic comparison data that keeps `duration` but still removes `pdays`.

    `pdays` is removed in both cases because most rows use `-1`, meaning the customer was not previously contacted. That value dominates the column and makes it less reliable for this workflow.


In [ ]:
# Run the reusable cleaning pipeline for both report scenarios.

from src.data.run_processing import process_data

no_duration = process_data(RAW_DATA, CLEANED_NO_DURATION, keep_duration=False)
with_duration = process_data(RAW_DATA, CLEANED_WITH_DURATION, keep_duration=True)

no_duration.shape, with_duration.shape

The no-duration dataset has 15 columns, while the with-duration dataset has 16 columns. This confirms that the only difference between the two modeling scenarios is the presence of `duration`.


## Final Cleaned Data Preview

        We finish by checking the production-safe cleaned dataset. At this point, `deposit` is encoded as `0` for `no` and `1` for `yes`, while categorical predictors remain as text until the feature-engineering notebook.


In [18]:
# Inspect the cleaned production dataset.

no_duration.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1,0,unknown,1
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1,0,unknown,1
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1,0,unknown,1
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,1,0,unknown,1
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,2,0,unknown,1
